In [1]:
import firebase_admin
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()

True

### Initialize Firebase


In [2]:
service_account = {
    "type": os.getenv("FIREBASE_TYPE"),
    "project_id": os.getenv("FIREBASE_PROJECT_ID"),
    "private_key_id": os.getenv("FIREBASE_PRIVATE_KEY_ID"),
    "private_key": os.getenv("FIREBASE_PRIVATE_KEY").replace("\\n", "\n"),
    "client_email": os.getenv("FIREBASE_CLIENT_EMAIL"),
    "client_id": os.getenv("FIREBASE_CLIENT_ID"),
    "auth_uri": os.getenv("FIREBASE_AUTH_URI"),
    "token_uri": os.getenv("FIREBASE_TOKEN_URI"),
    "auth_provider_x509_cert_url": os.getenv("FIREBASE_AUTH_PROVIDER_X509_CERT_URL"),
    "client_x509_cert_url": os.getenv("FIREBASE_CLIENT_X509_CERT_URL"),
    "universe_domain": os.getenv("FIREBASE_UNIVERSE_DOMAIN"),
}

In [3]:
credential = firebase_admin.credentials.Certificate(service_account)

firebase_admin.initialize_app(credential, {
    'databaseURL': os.getenv("FIREBASE_DATABASE_URL"),
    'storageBucket': os.getenv("FIREBASE_STORAGE_BUCKET"),
})

### Upload Data


In [4]:
from firebase_admin import storage

bucket = storage.bucket()

In [5]:
image_folder_path = './products/images/'

In [6]:
from firebase_admin import db

products_collection = db.reference('products')

In [7]:
dataframe = pd.read_json('./products/products.jsonl', lines=True)
dataframe.head()

,name,category,description,ingredients,price,rating,image_path
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]",4.50,4.7,cappuccino.jpg
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",3.25,4.3,SavoryScone.webp
2,Latte,Coffee,"Smooth and creamy, our latte combines rich esp...","[Espresso, Steamed Milk, Milk Foam]",4.75,4.8,Latte.jpg
3,Chocolate Chip Biscotti,Bakery,"Crunchy and delightful, this chocolate chip bi...","[Flour, Sugar, Chocolate Chips, Eggs, Almonds,...",2.50,4.6,chocolat_biscotti.jpg
4,Espresso shot,Coffee,"A bold shot of rich espresso, our espresso is ...",[Espresso],2.00,4.9,Espresso_shot.webp


### Upload images to firebase storage


In [8]:
def upload_images(bucket, image_path):
    image_name = image_path.split('/')[-1]
    
    blob = bucket.blob(f'products_images/{image_name}')
    
    #upload the image
    blob.upload_from_filename(image_path)
    
    #Make image public
    blob.make_public()
    
    #return the public url
    return blob.public_url


### Upload json data into firebase realtime DB


In [9]:
for index, row in dataframe.iterrows():
    print(index, row["name"])
    
    image_path = os.path.join(image_folder_path, row["image_path"])
    
    image_url = upload_images(bucket, image_path)
    
    # Convert the row (which is a pandas Series) into a dictionary
    product_data = row.to_dict()
    
    # Remove the local image path from the data dictionary, as we are replacing it with the URL
    product_data.pop("image_path")
    product_data["image_url"] = image_url
    
    # Push the product data to the Firebase Realtime Database under the 'products_collection' reference
    products_collection.push().set(product_data)

0 Cappuccino
1 Jumbo Savory Scone
2 Latte
3 Chocolate Chip Biscotti
4 Espresso shot
5 Hazelnut Biscotti
6 Chocolate Croissant
7 Dark chocolate
8 Cranberry Scone
9 Croissant
10 Almond Croissant
11 Ginger Biscotti
12 Oatmeal Scone
13 Ginger Scone
14 Chocolate syrup
15 Hazelnut syrup
16 Carmel syrup
17 Sugar Free Vanilla syrup
